# VROOM-SBI quickstart: FITS cube inference

This notebook builds a small synthetic IQUV cube with a known Faraday-thin source,
writes it to FITS, runs VROOM-SBI polarimetric inference over the cube, and
compares the recovered RM map against the injected truth.

Pre-trained models are downloaded automatically from HuggingFace on first run.

**Injected source**
- Model: Faraday-thin
- RM = +25 rad/m²
- Fractional polarization p₀ = 0.30
- Intrinsic angle χ₀ = 0.5 rad
- Single-pixel point source at cube centre
- Per-channel noise σ = 0.02 (fractional polarization units), giving peak SNR = 15 per channel


In [1]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import astropy.units as u
from astropy.io import fits
from astropy.wcs import WCS

## 1. Load the training frequency grid

The model was trained on the frequency grid in `freq.txt` (127 channels,
1.0–2.0 GHz, VLA L-band).  The synthetic cube must use this same grid so
that λ² values fed to the network match those seen during training.

In [2]:
freq_file = Path("..") / "freq.txt"
freqs_hz = np.loadtxt(freq_file, usecols=0)   # Hz, shape (127,)
n_freq = len(freqs_hz)
print(f"Frequency channels: {n_freq}")
print(f"Range: {freqs_hz[0]/1e9:.3f} – {freqs_hz[-1]/1e9:.3f} GHz")

Frequency channels: 128
Range: 1.000 – 2.000 GHz


## 2. Simulate the IQUV cube

In [3]:
import sys
sys.path.insert(0, str(Path("..")))

from src.simulator.base_simulator import RMSimulator
from src.simulator.physics import freq_to_lambda_sq

# Cube geometry
nx, ny = 64, 64
cx, cy = nx // 2, ny // 2  # source pixel

# Injected parameters (Faraday-thin, single component)
RM_TRUE   = 25.0   # rad/m²
P0_TRUE   = 0.30   # fractional polarization
CHI0_TRUE = 0.5    # rad
NOISE_SIGMA     = 0.02  # per-channel rms noise in fractional polarization units
TOTAL_INTENSITY = 1.0   # Jy (flat spectrum)

peak_snr = P0_TRUE / NOISE_SIGMA
print(f"Peak fractional polarization: {P0_TRUE:.2f}")
print(f"Per-channel noise sigma:      {NOISE_SIGMA:.3f}")
print(f"Peak SNR (per channel):       {peak_snr:.1f}")

# Simulate the noiseless QU spectrum
sim = RMSimulator(str(freq_file), n_components=1, model_type="faraday_thin")
theta = np.array([[RM_TRUE, P0_TRUE, CHI0_TRUE]])
qu_noiseless = sim.simulate_noiseless(theta)   # shape (2*n_freq,)
Q_src = qu_noiseless[:n_freq]
U_src = qu_noiseless[n_freq:]

# Build cubes
rng = np.random.default_rng(42)
I_cube = np.full((n_freq, ny, nx), TOTAL_INTENSITY, dtype=np.float32)
Q_cube = np.zeros((n_freq, ny, nx), dtype=np.float32)
U_cube = np.zeros((n_freq, ny, nx), dtype=np.float32)
V_cube = np.zeros((n_freq, ny, nx), dtype=np.float32)

# Point source: inject only at (cy, cx)
Q_cube[:, cy, cx] = Q_src
U_cube[:, cy, cx] = U_src

# Add noise everywhere
Q_cube += rng.normal(0, NOISE_SIGMA, Q_cube.shape).astype(np.float32)
U_cube += rng.normal(0, NOISE_SIGMA, U_cube.shape).astype(np.float32)
I_cube += rng.normal(0, NOISE_SIGMA * 0.1, I_cube.shape).astype(np.float32)

# Single-pixel spatial mask: only run inference on the source pixel
pixel_mask = np.zeros((ny, nx), dtype=bool)
pixel_mask[cy, cx] = True
# Keep source_mask as an alias for downstream cells
source_mask = pixel_mask

print(f"\nCentre pixel Q (ch 0): {Q_cube[0, cy, cx]:.4f}  (noiseless: {Q_src[0]:.4f})")
print(f"Centre pixel U (ch 0): {U_cube[0, cy, cx]:.4f}  (noiseless: {U_src[0]:.4f})")
print(f"Off-source rms Q:      {np.std(Q_cube[0, ~source_mask]):.4f}")


Peak fractional polarization: 0.30
Per-channel noise sigma:      0.020
Peak SNR (per channel):       15.0

Centre pixel Q (ch 0): 0.2194  (noiseless: 0.2113)
Centre pixel U (ch 0): -0.2228  (noiseless: -0.2130)
Off-source rms Q:      0.0200


## 3. Write FITS cubes with L-band WCS

In [4]:
cube_dir = Path("synthetic_cubes")
cube_dir.mkdir(exist_ok=True)

def make_wcs_header(freqs_hz, nx, ny):
    """Build a minimal FITS header with spatial + spectral WCS."""
    w = WCS(naxis=3)
    # Spectral axis (FREQ)
    w.wcs.ctype = ["RA---SIN", "DEC--SIN", "FREQ"]
    w.wcs.crpix = [nx / 2 + 1, ny / 2 + 1, 1]
    w.wcs.crval = [180.0, 40.0, freqs_hz[0]]
    w.wcs.cdelt = [-1.0 / 3600, 1.0 / 3600, freqs_hz[1] - freqs_hz[0]]
    w.wcs.cunit = ["deg", "deg", "Hz"]
    return w.to_header()

hdr = make_wcs_header(freqs_hz, nx, ny)

def write_stokes_cube(data, path, header, stokes_label):
    h = header.copy()
    h["BUNIT"] = "Jy/beam"
    h["STOKES"] = stokes_label
    fits.writeto(path, data.astype(np.float32), h, overwrite=True)
    print(f"Wrote {path}  shape={data.shape}")

write_stokes_cube(I_cube, cube_dir / "I.fits", hdr, "I")
write_stokes_cube(Q_cube, cube_dir / "Q.fits", hdr, "Q")
write_stokes_cube(U_cube, cube_dir / "U.fits", hdr, "U")
write_stokes_cube(V_cube, cube_dir / "V.fits", hdr, "V")

Wrote synthetic_cubes/I.fits  shape=(128, 64, 64)
Wrote synthetic_cubes/Q.fits  shape=(128, 64, 64)
Wrote synthetic_cubes/U.fits  shape=(128, 64, 64)
Wrote synthetic_cubes/V.fits  shape=(128, 64, 64)


## 4. Quick look at the cube

In [5]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

mid = n_freq // 2
axes[0].imshow(I_cube[mid], origin="lower", cmap="inferno")
axes[0].set_title(f"Stokes I  (ch {mid}, {freqs_hz[mid]/1e9:.2f} GHz)")
axes[0].set_xlabel("RA pixel")
axes[0].set_ylabel("Dec pixel")

im = axes[1].imshow(Q_cube[mid], origin="lower", cmap="RdBu_r",
                    vmin=-0.15, vmax=0.15)
axes[1].set_title("Stokes Q")
axes[1].set_xlabel("RA pixel")
plt.colorbar(im, ax=axes[1], label="Jy/beam")

im = axes[2].imshow(U_cube[mid], origin="lower", cmap="RdBu_r",
                    vmin=-0.15, vmax=0.15)
axes[2].set_title("Stokes U")
axes[2].set_xlabel("RA pixel")
plt.colorbar(im, ax=axes[2], label="Jy/beam")

plt.tight_layout()
plt.savefig("cube_preview.png", dpi=120, bbox_inches="tight")
plt.show()

# Also plot the QU spectrum at the source centre
fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(freqs_hz / 1e9, Q_cube[:, cy, cx], label="Q (source centre)", color="steelblue")
ax.plot(freqs_hz / 1e9, U_cube[:, cy, cx], label="U (source centre)", color="coral")
ax.axhline(0, color="k", lw=0.5)
ax.set_xlabel("Frequency (GHz)")
ax.set_ylabel("Jy/beam")
ax.set_title(f"QU spectrum at source centre  (RM = {RM_TRUE} rad/m²)")
ax.legend()
plt.tight_layout()
plt.savefig("qu_spectrum.png", dpi=120, bbox_inches="tight")
plt.show()

/tmp/ipykernel_275883/105234752.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_275883/105234752.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Load models (auto-downloads from HuggingFace if not present)

`engine.load_models()` has `auto_download=True` by default.
If `models/` is absent or empty, it fetches the pre-trained posteriors
from [huggingface.co/arpan-52/vroom-sbi](https://huggingface.co/arpan-52/vroom-sbi).
This only happens once; subsequent runs use the local cache.

In [6]:
from src.inference import InferenceEngine

engine = InferenceEngine(model_dir="../models", device="cpu")
engine.load_models()   # auto_download=True is the default
print("Loaded models:", list(engine.posteriors.keys()))

INFO: Loading models from ../models


INFO:   Loaded faraday_thin_n1


INFO: Loaded 1 posterior models


Loaded models: ['faraday_thin_n1']


## 6. Open FITS cubes and run masked inference

`run_inference_cube_chunked` accepts an explicit spatial `mask` (boolean array,
True = run inference).  Passing the single-pixel mask means inference runs on
exactly one pixel -- the injected point source -- regardless of the SNR threshold.


In [7]:
from src.io import open_qu_cubes_lazy, open_i_cube_lazy

q_cube, u_cube, shape, frequencies, wcs_2d = open_qu_cubes_lazy(
    str(cube_dir / "Q.fits"),
    str(cube_dir / "U.fits"),
)
i_cube = open_i_cube_lazy(str(cube_dir / "I.fits"))

print(f"Cube shape (freq, dec, ra): {shape}")
print(f"Frequency range: {frequencies[0]/1e9:.3f} – {frequencies[-1]/1e9:.3f} GHz")

INFO: Opened Q/U cubes lazily: shape (128, 64, 64), 128 channels


INFO: Opened Stokes I cube lazily: shape (128, 64, 64)


Cube shape (freq, dec, ra): (128, 64, 64)
Frequency range: 1.000 – 2.000 GHz


In [8]:
results = engine.run_inference_cube_chunked(
    q_cube,
    u_cube,
    shape=shape,
    frequencies_hz=frequencies,
    mask=pixel_mask,          # only process the injected point-source pixel
    snr_threshold=0.0,        # mask overrides SNR gating; set to 0 to be explicit
    # n_samples=200 keeps CPU runtime short on CPU; raise to 1000+ for production
    n_samples=200,
    i_cube=i_cube,
)
print("Output maps:", list(results.keys()))


INFO: Available RAM: 24.1 GB  →  chunk size: 64×64 (6 MB per chunk)


INFO: Pass 1a: estimating per-channel noise...


Pass 1a (rows):   0%|          | 0/1 [00:00<?, ?row/s]

Pass 1a (rows): 100%|██████████| 1/1 [00:00<00:00, 85.32row/s]


INFO: Weights: 0/128 channels flagged (weight=0)


...

/var/home/pjaganna/Software/vroom-sbi/.pixi/envs/notebooks/lib/python3.12/site-packages/nflows/transforms/lu.py:80: UserWarning: torch.triangular_solve is deprecated in favor of torch.linalg.solve_triangularand will be removed in a future PyTorch release.
torch.linalg.solve_triangular has its arguments reversed and does not return a copy of one of the inputs.
X = torch.triangular_solve(B, A).solution
should be replaced with
X = torch.linalg.solve_triangular(A, B). (Triggered internally at /home/conda/feedstock_root/build_artifacts/libtorch_1772252348570/work/aten/src/ATen/native/BatchLinearAlgebra.cpp:2264.)
  outputs, _ = torch.triangular_solve(


Pass 2 (rows): 100%|██████████| 1/1 [00:01<00:00,  1.36s/row]

Pass 2 (rows): 100%|██████████| 1/1 [00:01<00:00,  1.36s/row]


INFO: Chunked cube inference complete.


Output maps: ['rm_mean_comp1', 'amp_mean_comp1', 'chi0_mean_comp1', 'rm_std_comp1', 'amp_std_comp1', 'chi0_std_comp1', 'rm_p16_comp1', 'amp_p16_comp1', 'chi0_p16_comp1', 'rm_p84_comp1', 'amp_p84_comp1', 'chi0_p84_comp1', 'log_evidence', 'n_components']


## 7. Write output FITS maps

In [9]:
from src.io import write_results_maps

output_dir = Path("inference_output")
write_results_maps(results, wcs_2d, str(output_dir))

print("Written FITS maps:")
for f in sorted(output_dir.glob("*.fits")):
    print(" ", f.name)

INFO: Wrote 14 parameter maps and results.npz to inference_output


Written FITS maps:
  amp_mean_comp1.fits
  amp_p16_comp1.fits
  amp_p84_comp1.fits
  amp_std_comp1.fits
  chi0_mean_comp1.fits
  chi0_p16_comp1.fits
  chi0_p84_comp1.fits
  chi0_std_comp1.fits
  log_evidence.fits
  n_components.fits
  rm_mean_comp1.fits
  rm_p16_comp1.fits
  rm_p84_comp1.fits
  rm_std_comp1.fits


## 8. Compare recovered RM against injected truth

In [10]:
# Find the best-fit model key (highest coverage in results)
rm_keys = [k for k in results if k.startswith("rm_mean")]
print("RM maps available:", rm_keys)

rm_map  = results["rm_mean_comp1"]
rm_std  = results.get("rm_std_comp1", np.full_like(rm_map, np.nan))
amp_map = results.get("amp_mean_comp1", np.full_like(rm_map, np.nan))

# SNR map: peak polarized intensity / noise floor
# Compute per-pixel polarized intensity at the midpoint channel and the noise rms off-source
P_cube = np.sqrt(Q_cube**2 + U_cube**2)   # (n_freq, ny, nx)
P_peak = P_cube.max(axis=0)               # peak polarized intensity per pixel
snr_map = P_peak / NOISE_SIGMA

active  = np.isfinite(rm_map)
overlap = active & source_mask

print(f"Active (inferred) pixels:     {active.sum()}")
print(f"True source pixels (>5%):     {source_mask.sum()}")
print(f"Source pixels with inference: {overlap.sum()}")
print(f"Peak SNR (centre pixel):      {snr_map[cy, cx]:.1f}")
print(f"Median SNR over source:       {np.nanmedian(snr_map[source_mask]):.1f}")
print()
print(f"Recovered RM  mean ± std:  {np.nanmean(rm_map[overlap]):.2f} ± {np.nanstd(rm_map[overlap]):.2f} rad/m²")
print(f"Injected RM:               {RM_TRUE:.2f} rad/m²")
print(f"Median posterior σ_RM:     {np.nanmedian(rm_std[overlap]):.2f} rad/m²")


RM maps available: ['rm_mean_comp1']
Active (inferred) pixels:     1
True source pixels (>5%):     1
Source pixels with inference: 1
Peak SNR (centre pixel):      17.3
Median SNR over source:       17.3

Recovered RM  mean ± std:  25.47 ± 0.00 rad/m²
Injected RM:               25.00 rad/m²
Median posterior σ_RM:     0.82 rad/m²


In [11]:
fig, axes = plt.subplots(1, 4, figsize=(20, 4))

# Recovered RM map
im0 = axes[0].imshow(rm_map, origin="lower", cmap="RdBu_r",
                     vmin=RM_TRUE - 20, vmax=RM_TRUE + 20)
axes[0].set_title("Recovered RM (rad/m²)")
plt.colorbar(im0, ax=axes[0], label="rad/m²")

# RM uncertainty
im1 = axes[1].imshow(rm_std, origin="lower", cmap="viridis", vmin=0, vmax=10)
axes[1].set_title("RM uncertainty σ (rad/m²)")
plt.colorbar(im1, ax=axes[1], label="rad/m²")

# Residual
residual = np.where(overlap, rm_map - RM_TRUE, np.nan)
im2 = axes[2].imshow(residual, origin="lower", cmap="RdBu_r", vmin=-5, vmax=5)
axes[2].set_title(f"Residual (recovered − {RM_TRUE} rad/m²)")
plt.colorbar(im2, ax=axes[2], label="rad/m²")

# SNR map
im3 = axes[3].imshow(snr_map, origin="lower", cmap="inferno", vmin=0, vmax=snr_map.max())
axes[3].set_title("Peak polarized SNR")
plt.colorbar(im3, ax=axes[3], label="SNR")

for ax in axes:
    ax.set_xlabel("RA pixel")
    ax.set_ylabel("Dec pixel")

plt.suptitle(
    f"VROOM-SBI cube inference  |  injected RM = {RM_TRUE} rad/m²  |  "
    f"recovered {np.nanmean(rm_map[overlap]):.1f} ± {np.nanstd(rm_map[overlap]):.1f} rad/m²  |  "
    f"peak SNR = {snr_map[cy, cx]:.1f}",
    y=1.02,
)
plt.tight_layout()
plt.savefig("rm_recovery.png", dpi=120, bbox_inches="tight")
plt.show()


/tmp/ipykernel_275883/3414972859.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9. Posterior for a single source pixel

In [12]:
import corner
from src.simulator.physics import freq_to_lambda_sq

# Build the observation vector for the source centre pixel
q_pix = Q_cube[:, cy, cx]
u_pix = U_cube[:, cy, cx]
i_pix = I_cube[:, cy, cx]

# Normalise by I (same as what the cube inference path does)
q_frac = q_pix / i_pix
u_frac = u_pix / i_pix
qu_obs = np.concatenate([q_frac, u_frac])

# Run single-pixel inference
result, all_results = engine.infer(qu_obs, n_samples=500)

print(f"Best model: {result.model_type}, {result.n_components} component(s)")
comp = result.components[0]
print(f"  RM  = {comp.rm_mean:.2f} ± {comp.rm_std:.2f} rad/m²  (true: {RM_TRUE})")
print(f"  p₀  = {np.mean(comp.samples[:, 1]):.3f} ± {np.std(comp.samples[:, 1]):.3f}  (true: {P0_TRUE})")
print(f"  χ₀  = {comp.chi0_mean:.3f} ± {comp.chi0_std:.3f} rad  (true: {CHI0_TRUE})")

INFO: Found 'auto' as default backend, checking available backends


INFO: Matplotlib is available, defining as default backend


INFO: arviz_base 1.1.0 available, exposing its functions as part of the `arviz` namespace


INFO: arviz_stats 1.1.0 available, exposing its functions as part of the `arviz` namespace


INFO: arviz_plots 1.1.0 available, exposing its functions as part of the `arviz` namespace


  0%|          | 0/500 [00:00<?, ?it/s]

Best model: faraday_thin, 1 component(s)
  RM  = 25.50 ± 0.71 rad/m²  (true: 25.0)
  p₀  = 0.266 ± 0.039  (true: 0.3)
  χ₀  = 0.503 ± 0.033 rad  (true: 0.5)


In [13]:
samples = comp.samples   # shape (5000, 3): [RM, p0, chi0]
labels  = [r"RM (rad/m²)", r"$p_0$", r"$\chi_0$ (rad)"]
truths  = [RM_TRUE, P0_TRUE, CHI0_TRUE]

fig = corner.corner(
    samples,
    labels=labels,
    truths=truths,
    truth_color="tab:red",
    quantiles=[0.16, 0.5, 0.84],
    show_titles=True,
    title_kwargs={"fontsize": 11},
)
fig.suptitle("Posterior for source centre pixel", y=1.02)
plt.savefig("corner_source_pixel.png", dpi=120, bbox_inches="tight")
plt.show()

/tmp/ipykernel_275883/1992056289.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 10. Reconstructed vs true QU spectra with posterior uncertainty

Using the posterior samples from the centre pixel, we reconstruct the predicted
QU spectra for each posterior draw and show the 1σ envelope against the observed
data and the noiseless truth.


In [14]:
lam2 = freq_to_lambda_sq(freqs_hz)   # (n_freq,)

# Reconstruct QU spectrum for each posterior sample
n_samp = len(samples)
Q_pred = np.zeros((n_samp, n_freq))
U_pred = np.zeros((n_samp, n_freq))

for k, (rm_s, p0_s, chi0_s) in enumerate(samples):
    phi = 2 * (rm_s * lam2 + chi0_s)
    Q_pred[k] = p0_s * np.cos(phi)
    U_pred[k] = p0_s * np.sin(phi)

Q_lo, Q_med, Q_hi = np.percentile(Q_pred, [16, 50, 84], axis=0)
U_lo, U_med, U_hi = np.percentile(U_pred, [16, 50, 84], axis=0)

# Observed (noisy) centre pixel
q_obs = Q_cube[:, cy, cx] / I_cube[:, cy, cx]
u_obs = U_cube[:, cy, cx] / I_cube[:, cy, cx]

# Noiseless truth
Q_true = Q_src
U_true = U_src

fig, (ax_q, ax_u) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

for ax, obs, lo, med, hi, true, label in [
    (ax_q, q_obs, Q_lo, Q_med, Q_hi, Q_true, "Q / I"),
    (ax_u, u_obs, U_lo, U_med, U_hi, U_true, "U / I"),
]:
    ax.errorbar(freqs_hz / 1e9, obs, yerr=NOISE_SIGMA,
                fmt='.', color='steelblue', alpha=0.4, ms=3, lw=0.8,
                label="Observed (noisy)")
    ax.plot(freqs_hz / 1e9, true, 'k-', lw=1.5, label="Injected truth")
    ax.plot(freqs_hz / 1e9, med, color='tab:orange', lw=1.5, label="Posterior median")
    ax.fill_between(freqs_hz / 1e9, lo, hi, color='tab:orange', alpha=0.3, label="1σ posterior")
    ax.set_ylabel(label)
    ax.legend(fontsize=9, loc="upper right")
    ax.axhline(0, color='gray', lw=0.5, ls='--')

ax_u.set_xlabel("Frequency (GHz)")
fig.suptitle(
    f"Reconstructed QU spectra — centre pixel\n"
    f"RM = {comp.rm_mean:.1f} ± {comp.rm_std:.1f} rad/m²  "
    f"(true {RM_TRUE})   SNR = {snr_map[cy, cx]:.1f}",
    fontsize=11,
)
plt.tight_layout()
plt.savefig("qu_spectrum_reconstruction.png", dpi=120, bbox_inches="tight")
plt.show()


/tmp/ipykernel_275883/3117448241.py:49: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
